# Final Merchant Recommendations

This notebook presents the final merchant recommendations for BNPL onboarding.

The ranking methodology was developed and validated in `ranking_summary.ipynb`. This notebook focuses on the final decision outputs: validating the reliability of the selected merchants, presenting the final Top 100 merchants, and identifying the Top 10 merchants within each industry segment.

The final ranking combines:

- Merchant value
- Customer strength
- Revenue growth
- Revenue stability
- Market / regional context
- Fraud-risk safety

The baseline final score uses the validated weighting framework developed in the ranking methodology notebook.

## 1. Load Final Ranking Base

In [1]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()

if (cwd / "member5_ranking").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "member5_ranking").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the repository root containing member5_ranking."
    )

RANKING_PATH = (
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "ranking_analysis_base.csv"
)

ranking_df = pd.read_csv(RANKING_PATH)

print("Shape:", ranking_df.shape)
print("Unique merchants:", ranking_df["merchant_abn"].nunique())

ranking_df.head()

Shape: (4026, 86)
Unique merchants: 4026


,merchant_abn,merchant_name,merchant_category,merchant_pricing_level,merchant_take_rate_pct,has_merchant_master_record,total_transactions,total_revenue,avg_transaction_value,unique_consumers,...,rank_change_after_fraud,fraud_5_score,fraud_5_rank,fraud_10_score,fraud_10_rank,fraud_15_score,fraud_15_rank,fraud_weight_best_rank,fraud_weight_worst_rank,fraud_weight_rank_range
0,10023283211,Felis Limited,"furniture, home furnishings and equipment shop...",E,0.18,True,3261,703277.711451,215.663205,3032,...,-41,57.173903,1579,56.375921,1605,55.577938,1636,1579,1636,57
1,10142254217,Arcu Ac Orci Corporation,"cable, satellite, and other pay television and...",B,4.22,True,3036,118356.146073,38.984238,2849,...,30,61.559405,1323,61.228306,1306,60.897207,1291,1291,1323,32
2,10165489824,Nunc Sed Company,"jewelry, watch, clock, and silverware shops",B,4.40,True,5,56180.473857,11236.094771,5,...,253,26.506349,3500,30.374436,3351,34.242523,3186,3186,3500,314
3,10187291046,Ultricies Dignissim Lacus Foundation,"watch, clock, and jewelry repair shops",B,3.29,True,336,39693.730387,118.136102,335,...,235,43.765853,2457,46.120872,2330,48.475892,2162,2162,2457,295
4,10192359162,Enim Condimentum PC,"music shops - musical instruments, pianos, and...",A,6.33,True,385,177980.505456,462.287027,383,...,-245,57.830456,1544,55.300779,1678,52.771103,1844,1544,1844,300


## 2. Final Ranking Reliability Audit

Before presenting the final recommendations, the Top 100 merchants are checked for three potential reliability concerns:

1. whether growth estimates are based on insufficient transaction history,
2. whether external Census / SEIFA / ATO coverage is adequate, and
3. whether fraud-risk evidence is sufficiently available.

These checks do not directly change the ranking. They are used to assess how much confidence should be placed in the final recommendations.

In [2]:
final_top100_df = (
    ranking_df
    .sort_values("final_rank")
    .head(100)
    .copy()
)

print("Final Top 100 merchants:", len(final_top100_df))

Final Top 100 merchants: 100


In [3]:
growth_reliability = (
    final_top100_df["low_sample_growth_estimate"]
    .value_counts(dropna=False)
    .rename_axis("low_sample_growth_estimate")
    .to_frame("merchant_count")
)

growth_reliability

,merchant_count
low_sample_growth_estimate,
False,100


In [4]:
low_sample_count = final_top100_df["low_sample_growth_estimate"].fillna(False).sum()

print("Low-sample growth merchants:", low_sample_count)
print("Share of Top 100:", low_sample_count / 100)

Low-sample growth merchants: 0
Share of Top 100: 0.0


In [5]:
external_coverage_summary = (
    final_top100_df[
        "regional_data_coverage_rate_all_sources_by_count"
    ]
    .describe()
)

external_coverage_summary

count    100.000000
mean       0.808117
std        0.004085
min        0.795610
25%        0.806271
50%        0.807828
75%        0.809873
max        0.826344
Name: regional_data_coverage_rate_all_sources_by_count, dtype: float64

In [6]:
coverage_checks = pd.Series({
    "coverage_below_90pct":
        (
            final_top100_df[
                "regional_data_coverage_rate_all_sources_by_count"
            ] < 0.90
        ).sum(),

    "coverage_below_80pct":
        (
            final_top100_df[
                "regional_data_coverage_rate_all_sources_by_count"
            ] < 0.80
        ).sum(),

    "coverage_below_70pct":
        (
            final_top100_df[
                "regional_data_coverage_rate_all_sources_by_count"
            ] < 0.70
        ).sum(),
})

coverage_checks

coverage_below_90pct    100
coverage_below_80pct      1
coverage_below_70pct      0
dtype: int64

In [9]:
overall_external_coverage = (
    ranking_df[
        "regional_data_coverage_rate_all_sources_by_count"
    ]
    .describe()
)

top100_external_coverage = (
    final_top100_df[
        "regional_data_coverage_rate_all_sources_by_count"
    ]
    .describe()
)

coverage_comparison = pd.DataFrame({
    "all_eligible_merchants": overall_external_coverage,
    "final_top_100": top100_external_coverage,
})

coverage_comparison

,all_eligible_merchants,final_top_100
count,4026.000000,100.000000
mean,0.806718,0.808117
std,0.054179,0.004085
min,0.000000,0.795610
25%,0.796875,0.806271
50%,0.807871,0.807828
75%,0.819749,0.809873
max,1.000000,0.826344


In [10]:
overall_mean_coverage = ranking_df[
    "regional_data_coverage_rate_all_sources_by_count"
].mean()

top100_mean_coverage = final_top100_df[
    "regional_data_coverage_rate_all_sources_by_count"
].mean()

print("All eligible merchants mean coverage:", overall_mean_coverage)
print("Top 100 mean coverage:", top100_mean_coverage)
print("Difference:", top100_mean_coverage - overall_mean_coverage)

All eligible merchants mean coverage: 0.806718209745659
Top 100 mean coverage: 0.8081167967166142
Difference: 0.0013985869709551846


External-data coverage is highly consistent among the final Top 100 merchants. Their mean joint coverage across Census, SEIFA, and ATO sources is 80.81%, which is very close to the 80.67% average across all eligible merchants.

This suggests that the final recommendations are not disproportionately driven by merchants with poor external-data coverage.

In [7]:
fraud_evidence_summary = (
    final_top100_df["fraud_evidence_status"]
    .value_counts(dropna=False)
    .to_frame("merchant_count")
)

fraud_evidence_summary

,merchant_count
fraud_evidence_status,
consumer_only,86
consumer_and_direct,14


In [8]:
print(
    "Top 100 with direct merchant fraud evidence:",
    final_top100_df[
        "has_direct_merchant_fraud_information"
    ].fillna(False).sum()
)

print(
    "Top 100 with consumer fraud information:",
    final_top100_df[
        "has_consumer_risk_information"
    ].fillna(False).sum()
)

Top 100 with direct merchant fraud evidence: 14
Top 100 with consumer fraud information: 100


### Reliability Audit Interpretation

The final Top 100 shows strong reliability across the main diagnostic checks.

None of the selected merchants is flagged as having a low-sample growth estimate, indicating that the growth component is not being driven by merchants with insufficient transaction history.

External-data coverage is also consistent. The Top 100 has an average joint Census, SEIFA, and ATO coverage of 80.81%, compared with 80.67% across all eligible merchants, suggesting that the final recommendations are not disproportionately affected by poor external-data coverage.

All Top 100 merchants have consumer-level fraud information. Fourteen merchants additionally have direct merchant fraud evidence, while the remaining 86 rely on consumer-risk exposure only. Therefore, fraud risk is available for all selected merchants, although direct merchant-level evidence remains relatively sparse.

## 3. Locked Final Scoring Framework

Following feature construction, business-weight sensitivity analysis, fraud integration, fraud-weight sensitivity analysis, and reliability checks, the final ranking framework is fixed before producing the recommendation lists.

The final score is:

\[
\text{Final Score}
=
0.90 \times \text{Business Score}
+
0.10 \times \text{Fraud-Risk Safety Score}
\]

where:

\[
\text{Business Score}
=
0.30 \times \text{Value}
+
0.25 \times \text{Customer Strength}
+
0.20 \times \text{Growth}
+
0.15 \times \text{Stability}
+
0.10 \times \text{Market Context}
\]

This is equivalent to the following effective final weights:

- **Merchant Value: 27%**
- **Customer Strength: 22.5%**
- **Growth: 18%**
- **Stability: 13.5%**
- **Market / Regional Context: 9%**
- **Fraud-Risk Safety: 10%**

The weighting framework prioritises directly observed commercial performance while retaining supporting signals for future growth, consistency, socioeconomic context, and fraud risk.

Reliability checks indicate that:

- none of the final Top 100 merchants has a low-sample growth estimate;
- external Census, SEIFA, and ATO coverage among the Top 100 is consistent with the wider eligible merchant pool; and
- all Top 100 merchants have consumer-level fraud information, with a subset also supported by direct merchant-level fraud evidence.

The scoring framework is therefore treated as fixed for the final recommendation stage.

In [11]:
required_final_columns = [
    "final_score",
    "final_rank",
    "balanced_score",
    "risk_safety_score",
    "value_score",
    "customer_score",
    "growth_score",
    "stability_score",
    "market_score",
]

missing_final_columns = [
    col for col in required_final_columns
    if col not in ranking_df.columns
]

print("Missing required final columns:", missing_final_columns)

Missing required final columns: []


## 4. Final Top 100 Merchants

The final Top 100 merchants are selected using the locked final scoring framework.

These merchants represent the strongest overall onboarding candidates after considering commercial value, customer strength, growth, stability, market context, and fraud-risk safety.

In [12]:
final_top100 = (
    ranking_df
    .sort_values("final_rank")
    .head(100)
    .copy()
)

final_top100[
    [
        "final_rank",
        "merchant_abn",
        "merchant_name",
        "merchant_category",
        "final_score",
        "balanced_score",
        "risk_safety_score",
        "value_score",
        "customer_score",
        "growth_score",
        "stability_score",
        "market_score",
    ]
].head(20)

,final_rank,merchant_abn,merchant_name,merchant_category,final_score,balanced_score,risk_safety_score,value_score,customer_score,growth_score,stability_score,market_score
1766,1,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses",83.622818,85.327189,68.283481,99.975161,99.428713,59.119184,89.997512,51.539990
3614,2,90568944804,Diam Eu Dolor LLC,tent and awning shops,83.367001,87.168863,49.150238,99.230005,93.169399,75.167952,82.483205,67.014406
1401,3,40515428545,Elit Sed Consequat Associates,artist supply and craft shops,83.259317,84.659810,70.654883,99.652260,94.585196,50.684250,94.700174,67.759563
3881,4,96680767841,Ornare Limited,motor vehicle supplies and new parts,83.197503,85.723766,60.461138,99.850969,98.435171,63.772083,90.097039,48.907104
1838,5,50315283629,Iaculis Aliquet Diam LLC,"lawn and garden supply outlets, including nurs...",82.699138,84.213447,69.070360,97.615499,98.360656,57.302812,88.678776,55.762543
803,6,27326652377,Tellus Aenean Corporation,"music shops - musical instruments, pianos, and...",82.620053,86.253316,49.920689,99.403875,89.195231,75.267479,86.165713,61.549925
519,7,21439773999,Mauris Non Institute,"cable, satellite, and other pay television and...",82.565684,84.409979,65.967029,99.826130,99.726776,52.077631,92.062702,53.055142
348,8,17488304283,Posuere Cubilia Curae Corporation,"cable, satellite, and other pay television and...",82.545465,85.080227,59.732608,97.913562,98.559364,64.170192,93.480965,42.101341
987,9,31385641294,Semper Auctor PC,motor vehicle supplies and new parts,82.527965,84.426294,65.443009,99.130651,83.457526,69.793481,83.378950,73.571783
1214,10,35909341340,Arcu Sed Eu Incorporated,"computer programming , data processing, and in...",82.484230,84.237457,66.705189,99.627422,98.882265,53.670067,90.345857,53.427720


In [13]:
top20_inspection = final_top100[
    [
        "final_rank",
        "merchant_abn",
        "merchant_name",
        "merchant_category",
        "final_score",
        "balanced_score",
        "risk_safety_score",
        "value_score",
        "customer_score",
        "growth_score",
        "stability_score",
        "market_score",
        "fraud_evidence_status",
    ]
].head(20).copy()

top20_inspection.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "final_top20_inspection.csv",
    index=False
)

top20_inspection

,final_rank,merchant_abn,merchant_name,merchant_category,final_score,balanced_score,risk_safety_score,value_score,customer_score,growth_score,stability_score,market_score,fraud_evidence_status
1766,1,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses",83.622818,85.327189,68.283481,99.975161,99.428713,59.119184,89.997512,51.539990,consumer_and_direct
3614,2,90568944804,Diam Eu Dolor LLC,tent and awning shops,83.367001,87.168863,49.150238,99.230005,93.169399,75.167952,82.483205,67.014406,consumer_and_direct
1401,3,40515428545,Elit Sed Consequat Associates,artist supply and craft shops,83.259317,84.659810,70.654883,99.652260,94.585196,50.684250,94.700174,67.759563,consumer_only
3881,4,96680767841,Ornare Limited,motor vehicle supplies and new parts,83.197503,85.723766,60.461138,99.850969,98.435171,63.772083,90.097039,48.907104,consumer_and_direct
1838,5,50315283629,Iaculis Aliquet Diam LLC,"lawn and garden supply outlets, including nurs...",82.699138,84.213447,69.070360,97.615499,98.360656,57.302812,88.678776,55.762543,consumer_and_direct
803,6,27326652377,Tellus Aenean Corporation,"music shops - musical instruments, pianos, and...",82.620053,86.253316,49.920689,99.403875,89.195231,75.267479,86.165713,61.549925,consumer_only
519,7,21439773999,Mauris Non Institute,"cable, satellite, and other pay television and...",82.565684,84.409979,65.967029,99.826130,99.726776,52.077631,92.062702,53.055142,consumer_and_direct
348,8,17488304283,Posuere Cubilia Curae Corporation,"cable, satellite, and other pay television and...",82.545465,85.080227,59.732608,97.913562,98.559364,64.170192,93.480965,42.101341,consumer_only
987,9,31385641294,Semper Auctor PC,motor vehicle supplies and new parts,82.527965,84.426294,65.443009,99.130651,83.457526,69.793481,83.378950,73.571783,consumer_only
1214,10,35909341340,Arcu Sed Eu Incorporated,"computer programming , data processing, and in...",82.484230,84.237457,66.705189,99.627422,98.882265,53.670067,90.345857,53.427720,consumer_and_direct


In [15]:
final_top20 = (
    final_top100
    .head(20)
    .copy()
)

print("Final Top 20 merchants:", len(final_top20))

Final Top 20 merchants: 20


In [16]:
top20_score_summary = (
    final_top20[
        [
            "final_score",
            "balanced_score",
            "risk_safety_score",
            "value_score",
            "customer_score",
            "growth_score",
            "stability_score",
            "market_score",
        ]
    ]
    .describe()
    .T
)

top20_score_summary

,count,mean,std,min,25%,50%,75%,max
final_score,20.0,82.569727,0.452743,81.984554,82.295659,82.465018,82.639824,83.622818
balanced_score,20.0,85.174020,0.906988,83.864363,84.422215,84.995492,85.734106,87.168863
risk_safety_score,20.0,59.131090,8.487712,42.374802,51.569227,59.800589,66.676297,70.654883
value_score,20.0,98.667412,1.741055,93.591654,98.491058,99.329359,99.726776,99.975161
customer_score,20.0,95.904123,4.682382,83.457526,93.504719,98.497268,99.105812,99.975161
growth_score,20.0,61.240358,7.315947,50.684250,55.449117,59.728788,65.999005,75.267479
stability_score,20.0,90.350834,4.202671,82.483205,87.664842,90.221448,93.592934,98.283155
market_score,20.0,57.970691,8.886951,42.101341,51.136364,56.544958,67.200695,73.571783


In [17]:
score_dimensions = [
    "value_score",
    "customer_score",
    "growth_score",
    "stability_score",
    "market_score",
    "risk_safety_score",
]

top20_weakest_dimension = final_top20[
    ["final_rank", "merchant_name"] + score_dimensions
].copy()

top20_weakest_dimension["weakest_dimension"] = (
    top20_weakest_dimension[score_dimensions].idxmin(axis=1)
)

top20_weakest_dimension["weakest_score"] = (
    top20_weakest_dimension[score_dimensions].min(axis=1)
)

top20_weakest_dimension[
    [
        "final_rank",
        "merchant_name",
        "weakest_dimension",
        "weakest_score",
    ]
]

,final_rank,merchant_name,weakest_dimension,weakest_score
1766,1,Dignissim Maecenas Foundation,market_score,51.539990
3614,2,Diam Eu Dolor LLC,risk_safety_score,49.150238
1401,3,Elit Sed Consequat Associates,growth_score,50.684250
3881,4,Ornare Limited,market_score,48.907104
1838,5,Iaculis Aliquet Diam LLC,market_score,55.762543
803,6,Tellus Aenean Corporation,risk_safety_score,49.920689
519,7,Mauris Non Institute,growth_score,52.077631
348,8,Posuere Cubilia Curae Corporation,market_score,42.101341
987,9,Semper Auctor PC,risk_safety_score,65.443009
1214,10,Arcu Sed Eu Incorporated,market_score,53.427720


### Top 20 Profile Check

The Top 20 merchants are consistently strong in the core commercial dimensions of merchant value, customer strength, and revenue stability.

Variation is mainly concentrated in growth, market context, and fraud-risk safety. No Top 20 merchant exhibits an extreme weakness in the core commercial dimensions, suggesting that the highest-ranked merchants are supported by broad business strength rather than a single dominant metric.

In [18]:
final_top100.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "final_top_100.csv",
    index=False
)

print("Final Top 100 exported successfully.")

Final Top 100 exported successfully.


## 5. Segment-Level Top 10 Recommendations

To complement the overall Top 100 ranking, merchants are also compared within their broader industry segments.

The same locked final score is used for all segments. This preserves consistency across the recommendation framework while allowing strong merchants in smaller or structurally different industries to be identified.

For each segment, the Top 10 merchants are selected according to their final score.

In [19]:
segment_candidates = [
    col for col in ranking_df.columns
    if "segment" in col.lower()
    or "industry" in col.lower()
    or "group" in col.lower()
]

segment_candidates

[]

In [20]:
MAPPING_PATH = (
    PROJECT_ROOT
    / "member3_industry_growth"
    / "results"
    / "category_to_group_mapping.csv"
)

segment_mapping = pd.read_csv(MAPPING_PATH)

print("Mapping shape:", segment_mapping.shape)
segment_mapping.head()

Mapping shape: (25, 2)


,merchant_category,industry_group
0,"antique shops - sales, repairs, and restoratio...","Art, Gifts, Jewellery & Fashion"
1,art dealers and galleries,"Art, Gifts, Jewellery & Fashion"
2,artist supply and craft shops,"Creative, Books & Leisure"
3,bicycle shops - sales and service,"Mobility, Health & Specialist Services"
4,"books, periodicals, and newspapers","Creative, Books & Leisure"


In [21]:
segment_mapping.columns.tolist()

['merchant_category', 'industry_group']

In [22]:
ranking_with_segment = ranking_df.merge(
    segment_mapping,
    on="merchant_category",
    how="left",
    validate="many_to_one",
)

print("Before merge:", len(ranking_df))
print("After merge:", len(ranking_with_segment))
print(
    "Missing industry group:",
    ranking_with_segment["industry_group"].isna().sum()
)

ranking_with_segment["industry_group"].value_counts()

Before merge: 4026
After merge: 4026
Missing industry group: 0


industry_group
Digital, Technology & Communications      867
Home, Garden & Living                     827
Creative, Books & Leisure                 827
Mobility, Health & Specialist Services    806
Art, Gifts, Jewellery & Fashion           699
Name: count, dtype: int64

In [23]:
segment_top10 = (
    ranking_with_segment
    .sort_values(
        ["industry_group", "final_score"],
        ascending=[True, False]
    )
    .groupby("industry_group", group_keys=False)
    .head(10)
    .copy()
)

segment_top10["segment_rank"] = (
    segment_top10
    .groupby("industry_group")["final_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

segment_top10[
    [
        "industry_group",
        "segment_rank",
        "final_rank",
        "merchant_abn",
        "merchant_name",
        "merchant_category",
        "final_score",
        "balanced_score",
        "risk_safety_score",
    ]
].sort_values(
    ["industry_group", "segment_rank"]
)

,industry_group,segment_rank,final_rank,merchant_abn,merchant_name,merchant_category,final_score,balanced_score,risk_safety_score
3796,"Art, Gifts, Jewellery & Fashion",1,17,94493496784,Dictum Phasellus In Institute,"gift, card, novelty, and souvenir shops",82.212282,84.910756,57.926014
3450,"Art, Gifts, Jewellery & Fashion",2,21,86710922099,Ac Urna Consulting,art dealers and galleries,81.937033,82.541010,76.501246
2638,"Art, Gifts, Jewellery & Fashion",3,27,68559320474,Aliquam Auctor Associates,"antique shops - sales, repairs, and restoratio...",81.279774,84.968060,48.085203
125,"Art, Gifts, Jewellery & Fashion",4,37,12870663624,Vestibulum Ut Eros Corporation,"gift, card, novelty, and souvenir shops",80.936682,82.768985,64.445955
2293,"Art, Gifts, Jewellery & Fashion",5,40,60956456424,Ultricies Dignissim LLP,"gift, card, novelty, and souvenir shops",80.851119,84.398617,48.923635
1637,"Art, Gifts, Jewellery & Fashion",6,43,45629217853,Lacus Consulting,"gift, card, novelty, and souvenir shops",80.758010,83.962811,51.914797
3753,"Art, Gifts, Jewellery & Fashion",7,45,93558142492,Dolor Quisque Inc.,shoe shops,80.739429,82.680867,63.266485
65,"Art, Gifts, Jewellery & Fashion",8,46,11439466003,Blandit At LLC,shoe shops,80.725849,83.065984,59.664627
3957,"Art, Gifts, Jewellery & Fashion",9,47,98314397036,Lobortis Augue Industries,"gift, card, novelty, and souvenir shops",80.723914,84.544309,46.340358
3139,"Art, Gifts, Jewellery & Fashion",10,56,79417999332,Phasellus At Company,"gift, card, novelty, and souvenir shops",80.399094,84.712139,41.581690


In [24]:
print("Total selected merchants:", len(segment_top10))

segment_top10.groupby(
    "industry_group"
).size()

Total selected merchants: 50


industry_group
Art, Gifts, Jewellery & Fashion           10
Creative, Books & Leisure                 10
Digital, Technology & Communications      10
Home, Garden & Living                     10
Mobility, Health & Specialist Services    10
dtype: int64

In [25]:
print(
    "Unique merchants in segment Top 10:",
    segment_top10["merchant_abn"].nunique()
)

Unique merchants in segment Top 10: 50


### Segment-Level Summary

The segment-level recommendations use the same locked final score as the overall ranking.

The summary below compares the five industry groups using the average final score, average fraud-risk safety, and the range of overall ranks represented within each segment Top 10.

In [26]:
segment_summary = (
    segment_top10
    .groupby("industry_group")
    .agg(
        merchants_selected=("merchant_abn", "count"),
        mean_final_score=("final_score", "mean"),
        mean_balanced_score=("balanced_score", "mean"),
        mean_risk_safety_score=("risk_safety_score", "mean"),
        best_overall_rank=("final_rank", "min"),
        worst_overall_rank=("final_rank", "max"),
    )
    .sort_values("mean_final_score", ascending=False)
)

segment_summary

,merchants_selected,mean_final_score,mean_balanced_score,mean_risk_safety_score,best_overall_rank,worst_overall_rank
industry_group,,,,,,
"Home, Garden & Living",10,81.975835,84.877221,55.863358,2,42
"Digital, Technology & Communications",10,81.829747,84.587248,57.012237,7,33
"Mobility, Health & Specialist Services",10,81.748542,84.874692,53.613188,1,68
"Creative, Books & Leisure",10,81.464178,83.980183,58.820134,3,57
"Art, Gifts, Jewellery & Fashion",10,81.056319,83.855354,55.865001,17,56


In [27]:
segment_score_spread = (
    segment_top10
    .groupby("industry_group")["final_score"]
    .agg(["min", "median", "max"])
)

segment_score_spread["score_range"] = (
    segment_score_spread["max"]
    - segment_score_spread["min"]
)

segment_score_spread

,min,median,max,score_range
industry_group,,,,
"Art, Gifts, Jewellery & Fashion",80.399094,80.804564,82.212282,1.813188
"Creative, Books & Leisure",80.384797,81.122400,83.259317,2.874520
"Digital, Technology & Communications",81.107128,81.699541,82.565684,1.458556
"Home, Garden & Living",80.763509,82.065057,83.367001,2.603492
"Mobility, Health & Specialist Services",80.165606,81.899637,83.622818,3.457212


### Segment-Level Interpretation

The five industry segments show broadly similar performance among their Top 10 merchants. Mean final scores range from approximately 81.1 to 82.0, suggesting that the final scoring framework does not strongly favour a single industry group.

Within each segment, the Top 10 scores are also relatively concentrated, with score ranges of approximately 1.5 to 3.5 points.

All 50 segment-level Top 10 merchants also fall within the overall Top 100 ranking. Therefore, the segment-level recommendations do not introduce weaker merchants solely to achieve industry representation. Instead, they provide an industry-specific view of merchants that are already strong overall candidates.

In [28]:
segment_top10.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "segment_top_10.csv",
    index=False
)

segment_summary.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "segment_summary.csv"
)

print("Segment Top 10 and segment summary exported successfully.")

Segment Top 10 and segment summary exported successfully.
